# Power — Pushing a Block

Same set-up as the work-done demo: a constant force $F$ pushes a block of mass $m$ from rest along a frictionless floor.

**Power** is the rate of doing work:

$$P = \frac{W}{t}$$

measured in **watts** ($1\ \mathrm{W} = 1\ \mathrm{J\,s^{-1}}$). And since $W = Fs$ while the block moves with speed $v$, the **instantaneous** power can be written as

$$P = Fv$$

Press **Run** and watch the readout: the work $W = Fs$ accumulates, the average power $P = W/t$ grows, and the instantaneous power $P = Fv$ grows exactly in step with the speed. A constant force does **not** deliver constant power — the faster the block moves, the more joules per second it gains.

**Check:** the block starts from rest and $v$ grows steadily, so the average of $Fv$ over the run is exactly half its final value — the instantaneous power at the finish is always exactly **twice** the average power.

**Things to try:**

- Bigger $F$ → more work, delivered over a shorter time → much more power.
- Bigger $m$ → sluggish acceleration, so the same work is spread over a longer time → less power.


In [1]:
import numpy as np

import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle, FancyArrowPatch

from matplotlib.animation import FuncAnimation

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    %pip install -q ipywidgets
    import ipywidgets as widgets

from IPython.display import display, HTML


TRACK = 21.5   # length of the visible track the block travels (m)


def build_block_figure(x=0.0, F=100, m=20, a=5.0):
    """Draw the block, the force arrow and the acceleration arrow at position x."""
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.set_xlim(0, 25)
    ax.set_ylim(-1, 6)
    ax.axhline(1, color="black", linewidth=2)

    block = Rectangle((x, 1), 2.5, 1.6, facecolor="lightblue",
                      edgecolor="black", linewidth=2)
    ax.add_patch(block)
    mass_text = ax.text(x + 1.25, 1.8, f"{m} kg", ha="center", va="center",
                        fontsize=13)

    force_arrow = FancyArrowPatch((x + 2.7, 1.8), (x + 5.7, 1.8),
                                  arrowstyle="-|>", mutation_scale=20,
                                  linewidth=2, color="black")
    ax.add_patch(force_arrow)
    force_text = ax.text(x + 4.2, 2.25, f"F = {F} N", ha="center", fontsize=12)

    accel_arrow = FancyArrowPatch((x + 0.2, 4.1), (x + 4.2, 4.1),
                                  arrowstyle="-|>", mutation_scale=22,
                                  linewidth=2.5, color="firebrick")
    ax.add_patch(accel_arrow)
    accel_text = ax.text(x + 2.2, 4.65, f"a = {a:.2f} m/s²",
                         ha="center", fontsize=13, fontweight="bold")

    title = ax.set_title("", fontsize=12)

    ax.set_xlabel("Position (m)")
    ax.set_yticks([])
    for side in ("left", "right", "top"):
        ax.spines[side].set_visible(False)
    fig.tight_layout()

    artists = dict(
        block=block, mass_text=mass_text,
        force_arrow=force_arrow, force_text=force_text,
        accel_arrow=accel_arrow, accel_text=accel_text,
        title=title,
    )
    return fig, artists


def readout(F, m, t):
    """Distance, work, average power and instantaneous power after time t."""
    a = F / m
    s = 0.5 * a * t**2
    v = a * t
    W = F * s
    P_avg = W / t if t > 0 else 0.0      # W grows like t^2, so P_avg -> 0 at t = 0
    P_inst = F * v
    return a, s, v, W, P_avg, P_inst


def power_animation(F, m, n_frames=35):
    """In-browser animation: the block accelerates; P = W/t and P = Fv grow."""
    a = F / m
    t_end = np.sqrt(2 * TRACK / a)         # ends as the block reaches the edge
    times = np.linspace(0, t_end, n_frames)

    fig, art = build_block_figure(x=0.0, F=F, m=m, a=a)

    def update(i):
        t = times[i]
        a_, s, v, W, P_avg, P_inst = readout(F, m, t)
        art["block"].set_x(s)
        art["mass_text"].set_position((s + 1.25, 1.8))
        art["force_arrow"].set_positions((s + 2.7, 1.8), (s + 5.7, 1.8))
        art["force_text"].set_position((s + 4.2, 2.25))
        art["accel_arrow"].set_positions((s + 0.2, 4.1), (s + 4.2, 4.1))
        art["accel_text"].set_position((s + 2.2, 4.65))
        art["title"].set_text(
            f"t = {t:,.2f} s      W = Fs = {W:,.0f} J      "
            f"P = W/t = {P_avg:,.0f} W      P = Fv = {P_inst:,.0f} W")
        return tuple(art.values())

    anim = FuncAnimation(fig, update, frames=n_frames, interval=60,
                         repeat=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())


def show_start(F, m):
    """Static preview: block at the start line."""
    a = F / m
    fig, art = build_block_figure(x=0.0, F=F, m=m, a=a)
    art["title"].set_text("Press Run — then watch the power grow with speed")
    display(fig)
    plt.close(fig)


def summary(F, m):
    a = F / m
    t_end = np.sqrt(2 * TRACK / a)
    W_full = F * TRACK
    P_avg = W_full / t_end
    P_inst_end = F * a * t_end
    return (f"<b>F = {F} N</b>, m = {m} kg &nbsp;→&nbsp; a = F/m = "
            f"<b>{a:.2f} m/s²</b><br>"
            f"Full {TRACK:g} m track: W = Fs = <b>{W_full:,.0f} J</b> in "
            f"t = √(2s/a) = <b>{t_end:.2f} s</b><br>"
            f"Average power P = W/t = <b>{P_avg:,.0f} W</b> &nbsp;·&nbsp; "
            f"instantaneous power at the finish P = Fv = <b>{P_inst_end:,.0f} W</b> "
            f"(exactly double)")


In [2]:
# ============================================================
# Power: vary F and m, then watch P = W/t = Fv
# ============================================================

force_slider = widgets.IntSlider(
    value=100, min=20, max=300, step=20,
    description="Force F (N):",
    continuous_update=False
)

mass_slider = widgets.IntSlider(
    value=20, min=5, max=60, step=5,
    description="Mass m (kg):",
    continuous_update=False
)

run_button = widgets.Button(description="Run", button_style="primary", icon="play")
reset_button = widgets.Button(description="Reset")
value_label = widgets.HTML()
sim_output = widgets.Output(layout=widgets.Layout(width="700px"))


def show_initial():
    value_label.value = summary(force_slider.value, mass_slider.value)
    show_start(force_slider.value, mass_slider.value)


def run_clicked(button):
    value_label.value = summary(force_slider.value, mass_slider.value)
    with sim_output:
        sim_output.clear_output(wait=True)
        display(power_animation(force_slider.value, mass_slider.value))


def reset_clicked(button):
    with sim_output:
        sim_output.clear_output(wait=True)
        show_initial()


def slider_changed(change):
    with sim_output:
        sim_output.clear_output(wait=True)
        show_initial()


run_button.on_click(run_clicked)
reset_button.on_click(reset_clicked)
force_slider.observe(slider_changed, names="value")
mass_slider.observe(slider_changed, names="value")

controls = widgets.VBox([
    widgets.HTML("<h3>Controls</h3>"),
    force_slider,
    mass_slider,
    widgets.HBox([run_button, reset_button]),
    value_label
], layout=widgets.Layout(width="340px"))

display(widgets.HBox([controls, sim_output],
                     layout=widgets.Layout(align_items="flex-start", gap="18px")))

with sim_output:
    show_initial()


## Part 2 — The Delivery Deadline (fixed distance, set the time)

Now the job has a **deadline**: the block must be pushed the fixed 10 m **within the time you set** — "I want this crate delivered 10 m in 5 seconds" (or 10 seconds, or 2…).

Your chosen force decides whether the delivery makes it. The block accelerates at $a = F/m$, so the delivery actually takes

$$t = \sqrt{\frac{2s}{a}} = \sqrt{\frac{2sm}{F}}$$

The work is always $W = Fs = 10F$, and the average power of the delivery is

$$P = \frac{W}{t}$$

**Things to try:**

- Shorten the deadline → the same 10 m must arrive sooner → you need a bigger force, and the power shoots up.
- With the deadline and mass fixed, find the smallest force that just makes it: $F = \dfrac{2ms}{t^2}$.
- Bigger mass with the same deadline → more force needed → more work **and** more power — heavy deliveries on a tight schedule are expensive.

In [ ]:
DELIVERY = 10.0   # fixed delivery distance (m)


def build_deadline_figure(x=0.0, F=100, m=20, a=5.0):
    """Block, force and acceleration arrows, finish line at 10 m."""
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.set_xlim(0, 16)
    ax.set_ylim(-1, 6)
    ax.axhline(1, color="black", linewidth=2)

    ax.axvline(DELIVERY, color="green", linestyle="--", linewidth=2)
    ax.text(DELIVERY, 5.55, f"finish: s = {DELIVERY:g} m", color="green",
            ha="center", fontsize=11, fontweight="bold")

    block = Rectangle((x, 1), 2.5, 1.6, facecolor="lightblue",
                      edgecolor="black", linewidth=2)
    ax.add_patch(block)
    mass_text = ax.text(x + 1.25, 1.8, f"{m} kg", ha="center", va="center",
                        fontsize=13)

    force_arrow = FancyArrowPatch((x + 2.7, 1.8), (x + 5.7, 1.8),
                                  arrowstyle="-|>", mutation_scale=20,
                                  linewidth=2, color="black")
    ax.add_patch(force_arrow)
    force_text = ax.text(x + 4.2, 2.25, f"F = {F} N", ha="center", fontsize=12)

    accel_arrow = FancyArrowPatch((x + 0.2, 4.1), (x + 4.2, 4.1),
                                  arrowstyle="-|>", mutation_scale=22,
                                  linewidth=2.5, color="firebrick")
    ax.add_patch(accel_arrow)
    accel_text = ax.text(x + 2.2, 4.65, f"a = {a:.2f} m/s²",
                         ha="center", fontsize=13, fontweight="bold")

    title = ax.set_title("", fontsize=12)

    ax.set_xlabel("Position (m)")
    ax.set_yticks([])
    for side in ("left", "right", "top"):
        ax.spines[side].set_visible(False)
    fig.tight_layout()

    artists = dict(
        block=block, mass_text=mass_text,
        force_arrow=force_arrow, force_text=force_text,
        accel_arrow=accel_arrow, accel_text=accel_text,
        title=title,
    )
    return fig, artists


def deadline_animation(F, m, t_limit, n_frames=35):
    """Push with force F until the 10 m is covered or the deadline runs out."""
    a = F / m
    t_need = np.sqrt(2 * DELIVERY / a)       # time the delivery actually takes
    t_end = min(t_need, t_limit)             # clock stops at delivery or deadline
    times = np.linspace(0, t_end, n_frames)

    fig, art = build_deadline_figure(x=0.0, F=F, m=m, a=a)

    def update(i):
        t = times[i]
        s = 0.5 * a * t**2
        W = F * s
        P_avg = W / t if t > 0 else 0.0      # W grows like t², so P_avg -> 0 at t = 0
        art["block"].set_x(s)
        art["mass_text"].set_position((s + 1.25, 1.8))
        art["force_arrow"].set_positions((s + 2.7, 1.8), (s + 5.7, 1.8))
        art["force_text"].set_position((s + 4.2, 2.25))
        art["accel_arrow"].set_positions((s + 0.2, 4.1), (s + 4.2, 4.1))
        art["accel_text"].set_position((s + 2.2, 4.65))
        art["title"].set_text(
            f"t = {t:,.2f} / {t_limit:g} s      s = {s:,.1f} / {DELIVERY:g} m      "
            f"W = {W:,.0f} J      P = W/t = {P_avg:,.0f} W")
        return tuple(art.values())

    anim = FuncAnimation(fig, update, frames=n_frames, interval=60,
                         repeat=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())


def show_deadline_start(F, m, t_limit):
    """Static preview: block at the start line, finish line at 10 m."""
    a = F / m
    fig, art = build_deadline_figure(x=0.0, F=F, m=m, a=a)
    art["title"].set_text(f"Press Run — deliver {DELIVERY:g} m within {t_limit:g} s")
    display(fig)
    plt.close(fig)


def deadline_summary(F, m, t_limit):
    a = F / m
    t_need = np.sqrt(2 * DELIVERY / a)
    W_full = F * DELIVERY
    P_avg = W_full / t_need
    F_needed = 2 * m * DELIVERY / t_limit**2
    if t_need <= t_limit:
        verdict = (f"✓ Delivered in <b>{t_need:.2f} s</b> — inside the "
                   f"{t_limit:g} s deadline")
    else:
        verdict = (f"✗ Needs {t_need:.2f} s — misses the {t_limit:g} s deadline; "
                   f"you'd need F ≥ 2ms/t² = <b>{F_needed:,.0f} N</b>")
    return (f"<b>F = {F} N</b>, m = {m} kg, deadline = {t_limit:g} s "
            f"&nbsp;→&nbsp; a = F/m = <b>{a:.2f} m/s²</b><br>"
            f"{verdict}<br>"
            f"Delivery: W = Fs = ({F})({DELIVERY:g}) = <b>{W_full:,.0f} J</b>, so "
            f"P = W/t = <b>{P_avg:,.0f} W</b>")

In [ ]:
# ============================================================
# The delivery deadline: fixed 10 m — vary F, m and the time
# ============================================================

t_force_slider = widgets.IntSlider(
    value=60, min=10, max=300, step=10,
    description="Force F (N):",
    continuous_update=False
)

t_mass_slider = widgets.IntSlider(
    value=20, min=5, max=60, step=5,
    description="Mass m (kg):",
    continuous_update=False
)

t_time_slider = widgets.IntSlider(
    value=5, min=2, max=15, step=1,
    description="Deadline t (s):",
    continuous_update=False
)

t_run_button = widgets.Button(description="Run", button_style="primary", icon="play")
t_reset_button = widgets.Button(description="Reset")
t_value_label = widgets.HTML()
t_sim_output = widgets.Output(layout=widgets.Layout(width="700px"))


def t_show_initial():
    t_value_label.value = deadline_summary(t_force_slider.value,
                                           t_mass_slider.value,
                                           t_time_slider.value)
    show_deadline_start(t_force_slider.value, t_mass_slider.value,
                        t_time_slider.value)


def t_run_clicked(button):
    t_value_label.value = deadline_summary(t_force_slider.value,
                                           t_mass_slider.value,
                                           t_time_slider.value)
    with t_sim_output:
        t_sim_output.clear_output(wait=True)
        display(deadline_animation(t_force_slider.value, t_mass_slider.value,
                                   t_time_slider.value))


def t_reset_clicked(button):
    with t_sim_output:
        t_sim_output.clear_output(wait=True)
        t_show_initial()


def t_slider_changed(change):
    with t_sim_output:
        t_sim_output.clear_output(wait=True)
        t_show_initial()


t_run_button.on_click(t_run_clicked)
t_reset_button.on_click(t_reset_clicked)
t_force_slider.observe(t_slider_changed, names="value")
t_mass_slider.observe(t_slider_changed, names="value")
t_time_slider.observe(t_slider_changed, names="value")

t_controls = widgets.VBox([
    widgets.HTML("<h3>Controls — the delivery deadline</h3>"),
    t_force_slider,
    t_mass_slider,
    t_time_slider,
    widgets.HBox([t_run_button, t_reset_button]),
    t_value_label
], layout=widgets.Layout(width="340px"))

display(widgets.HBox([t_controls, t_sim_output],
                     layout=widgets.Layout(align_items="flex-start", gap="18px")))

with t_sim_output:
    t_show_initial()